# Day 064 — Exercise 5: Write Scaffold

The last step of Capstone Build I: write the MVP scaffold to disk. A scaffold is a working skeleton — the file structure a developer needs to start adding real logic. It should:

- Run without errors (even with TODO stubs)
- Include all dependency and deployment config
- Be self-documenting (each TODO says exactly what to add)

Generating it programmatically from a notebook uses the `repr()` source-embedding pattern from Day 051.

In [ ]:
from pathlib import Path

_APP_CONTENT = '"""app.py — AI Writing Assistant scaffold.\n\nReplace each TODO with real code following the pattern from Day 064.\n\nRun:  uvicorn app:app --reload\nDocs: http://localhost:8000/docs\n"""\nimport os\nfrom fastapi import FastAPI, HTTPException\nfrom pydantic import BaseModel, Field\n\nAPP_VER = "0.1.0"\n\n# TODO: import ollama for production usage\n# TODO: define DAILY_LIMITS, TEMPLATES, FEATURE_MATRIX\n\napp = FastAPI(title="AI Writing Assistant", version=APP_VER)\n\n\n@app.get("/health")\ndef health():\n    return {"status": "ok", "version": APP_VER}\n\n\n@app.get("/templates")\ndef list_templates():\n    # TODO: return {"templates": list(TEMPLATES.keys())}\n    return {"templates": []}\n\n\n@app.post("/generate")\ndef generate():\n    # TODO: add request model, rate-limit check, ollama call, store result\n    raise HTTPException(501, "Not implemented")\n\n\n@app.get("/history/{user_id}")\ndef history(user_id: str):\n    # TODO: return stored items for this user\n    return {"user_id": user_id, "count": 0, "items": []}\n'

SCAFFOLD_FILES = {
    "app.py":           _APP_CONTENT,
    "requirements.txt": "fastapi\nhttpx\nollama\nuvicorn[standard]\n",
    "Procfile":         "web: uvicorn app:app --host 0.0.0.0 --port $PORT\n",
}


## Task

Implement `write_scaffold(directory: str) -> list[str]`:

- Create `directory` if it doesn't exist (`mkdir(parents=True, exist_ok=True)`)
- Write each file from `SCAFFOLD_FILES` into the directory
- Return the list of filenames created (just names, not full paths)

## Your Implementation

In [ ]:
def write_scaffold(directory: str) -> list[str]:
    """Write MVP scaffold files to directory.

    Creates these files inside `directory`:
        app.py           — FastAPI scaffold with TODO stubs
        requirements.txt — package list (fastapi, ollama, uvicorn[standard], httpx)
        Procfile         — 'web: uvicorn app:app --host 0.0.0.0 --port $PORT'

    Returns the list of filenames created (just names, not full paths).
    """
    # TODO: create Path(directory), write each file, return list of names
    raise NotImplementedError


In [ ]:
def write_scaffold(directory: str) -> list[str]:
    base  = Path(directory)
    base.mkdir(parents=True, exist_ok=True)
    files = list(SCAFFOLD_FILES.keys())
    for filename, content in SCAFFOLD_FILES.items():
        (base / filename).write_text(content, encoding="utf-8")
    return files


## Automated checks

In [ ]:
score, total = 0, 5
try:
    import tempfile
    from pathlib import Path

    with tempfile.TemporaryDirectory() as tmpdir:
        created = write_scaffold(tmpdir)

        # returns a list
        assert isinstance(created, list) and len(created) > 0
        score += 1; print("\u2705 write_scaffold returns a non-empty list")

        # at least 3 files
        assert len(created) >= 3, f"Expected >=3 files, got {len(created)}"
        score += 1; print("\u2705 at least 3 files created")

        # all files actually exist
        for fname in created:
            assert (Path(tmpdir) / fname).exists(), f"{fname} not created"
        score += 1; print("\u2705 all listed filenames exist on disk")

        # requirements.txt contains fastapi
        req = next((f for f in created if "requirements" in f.lower()), None)
        assert req is not None, "requirements.txt not found in created list"
        req_text = (Path(tmpdir) / req).read_text()
        assert "fastapi" in req_text.lower()
        score += 1; print("\u2705 requirements.txt contains 'fastapi'")

        # Procfile contains uvicorn
        proc = next((f for f in created if "procfile" in f.lower()), None)
        assert proc is not None, "Procfile not found in created list"
        proc_text = (Path(tmpdir) / proc).read_text()
        assert "uvicorn" in proc_text
        score += 1; print("\u2705 Procfile contains 'uvicorn'")

except Exception as e:
    print(f"\u274c {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def write_scaffold(directory: str) -> list[str]:
    base  = Path(directory)
    base.mkdir(parents=True, exist_ok=True)
    files = list(SCAFFOLD_FILES.keys())
    for filename, content in SCAFFOLD_FILES.items():
        (base / filename).write_text(content, encoding="utf-8")
    return files
```

**Pattern**: `SCAFFOLD_FILES` is a dict of `{filename: content}`. Iterating it gives both at once. `exist_ok=True` makes the function idempotent — safe to call twice (overwrites files, doesn't error on existing directory). Return the keys (filenames), not the full paths — callers can reconstruct the full path themselves.

</details>